# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules (these will be checked in the model answer review next week)
1. Every bar chart **must have a zero baseline** — no exceptions (SWD p.51)
2. Every chart **must have an insight title**, not a topic title (SWD p.29)
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names (SWD p.57)

---


## Setup — Run this cell first


In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv('world_happiness_2023.csv')

# Rename to match what the exercise expects
df.columns = ['Country', 'Region', 'Happiness_Score', 'GDP', 'Social_Support',
              'Life_Expectancy', 'Freedom', 'Generosity', 'Corruption']

print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [11]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: Region, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda      

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [12]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # sort for horizontal bar

print(region_avg)

# Step 2: Build your chart
# YOUR CODE HERE


# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # ascending → happiest at top in horizontal bar

print(region_avg)

# Step 2: Build the chart
fig = px.bar(
    data_frame=region_avg,
    x='Happiness_Score',
    y='Region',
    orientation='h',
    color='Happiness_Score',
    color_continuous_scale='Blues',
    range_color=[2.5, 7.5],
    text=region_avg['Happiness_Score'].round(2),
    title='Western Europe leads global happiness by a wide margin — Sub-Saharan Africa and South Asia trail by over 2.5 points',
    labels={'Happiness_Score': '', 'Region': ''}
)

fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside',
    marker_line_width=1.5
)

fig.update_layout(
    xaxis=dict(range=[0, 8.5], showticklabels=False, showgrid=False),
    yaxis=dict(gridcolor='white'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    coloraxis_showscale=False,
    height=500,
    margin=dict(l=20, r=80, t=70, b=30)
)

fig.show()



                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533
                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [13]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score').reset_index(drop=True)
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

# Step 2: Build the chart
fig = px.bar(
    data_frame=combined,
    x='Happiness_Score',
    y='Country',
    orientation='h',
    color='Group',
    color_discrete_map={'Top 8': '#2E75B6', 'Bottom 8': '#E63946'},
    hover_data={'Region': True, 'Happiness_Score': ':.2f', 'Group': False},
    custom_data=combined[['Region']]
)

# Custom tooltip
fig.update_traces(
    hovertemplate='<b>%{y}</b><br>Score: %{x:.2f}<br>%{customdata[0]}<extra></extra>'
)

# Stretch goal — global average reference line
fig.add_vline(
    x=global_avg,
    line_dash='dash',
    line_color='#888888',
    line_width=1.5,
    annotation_text=f'Global avg: {global_avg:.2f}',
    annotation_position='top',
    annotation_font=dict(size=11, color='#888888')
)

# Gap annotation box
gap = top8['Happiness_Score'].max() - bottom8['Happiness_Score'].min()
fig.add_annotation(
    x=6.0, y=4,
    text=f'<b>Gap: {gap:.1f} points</b><br>separates the happiest<br>and least happy nations',
    showarrow=False,
    font=dict(size=11, color='#333333', family='Arial'),
    bgcolor='#FFFDE7',
    bordercolor='#CCCCCC',
    borderwidth=1,
    borderpad=5
)

# Group labels
fig.add_annotation(x=7.5, y=15, text='<b>Top 8</b>',
                   showarrow=False, font=dict(color='#2E75B6', size=12))
fig.add_annotation(x=3.5, y=0, text='<b>Bottom 8</b>',
                   showarrow=False, font=dict(color='#E63946', size=12))

fig.update_layout(
    title=f'The happiest and least happy nations are {gap:.1f} points apart — happiness inequality is vast',
    xaxis=dict(range=[0, 8.8], gridcolor='#EEEEEE', title='Happiness Score (0–10)'),
    yaxis=dict(gridcolor='white', title=''),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    showlegend=False,
    margin=dict(l=30, r=120, t=60, b=40),
    height=560
)

fig.show()

Global average: 5.81


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [14]:
# Stretch goal — grouped bar chart
# YOUR CODE HERE

# Stretch goal — grouped bar chart
# ---------------------------------

# Step 1: Filter to the 5 regions and compute averages
target_regions = ['Western Europe', 'Latin America and Caribbean',
                  'East Asia', 'Sub-Saharan Africa', 'South Asia']

region_factors = (df[df['Region'].isin(target_regions)]
                  .groupby('Region')[['GDP', 'Freedom']]
                  .mean()
                  .reset_index()
                  .sort_values('GDP', ascending=True))  # ascending → highest at top

print(region_factors)

# Step 2: Melt to long format for grouped bar
melted = region_factors.melt(id_vars='Region', var_name='Factor', value_name='Score')

# Step 3: Build the chart
fig = px.bar(
    data_frame=melted,
    x='Score',
    y='Region',
    color='Factor',
    barmode='group',
    orientation='h',
    color_discrete_map={'GDP': '#2E75B6', 'Freedom': '#F4A261'},
    category_orders={'Region': region_factors['Region'].tolist()},
    text=melted['Score'].round(2),
    title="Wealth Doesn't Buy Freedom: Western Europe leads on GDP but freedom gaps are smaller across regions",
    labels={'Score': '', 'Region': '', 'Factor': ''}
)

fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside'
)

fig.update_layout(
    xaxis=dict(range=[0, 13], showticklabels=False, showgrid=False),
    yaxis=dict(gridcolor='white'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(orientation='h', y=1.08, x=0, title=''),
    height=420,
    margin=dict(l=20, r=80, t=80, b=30)
)

fig.show()

                        Region        GDP   Freedom
3           Sub-Saharan Africa   7.082000  0.599429
2                   South Asia   7.152250  0.546250
1  Latin America and Caribbean   9.134923  0.754231
0                    East Asia  10.312000  0.754000
4               Western Europe  10.963533  0.896067
